In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
import os
list = os.listdir("drive/MyDrive/data")
list

['employee_salary_200.csv',
 'House-Prediction_Dataset',
 'corona prediction data',
 'titanic data set',
 'car_price_dataset.csv',
 'boxcox_practice_data.csv',
 'transform_practice_data.csv',
 'Movie-Recommendation-System',
 'house_price_missing.csv']

In [34]:
file = "/content/drive/MyDrive/data/house_price_missing.csv"

In [35]:
import pandas as pd
import numpy as np


In [36]:
df = pd.read_csv(file)
df.head()


,area_sqft,bedrooms,bathrooms,age_years,distance_km,garage_size,lot_size_sqft,quality_score,price
0,2098.0,4.0,4.0,23.1,5.69,3.8,2250.0,9.4,345686.0
1,1717.0,4.0,4.0,11.9,6.29,3.6,2402.0,8.0,315048.0
2,2188.6,3.0,3.0,2.8,5.58,3.2,3609.0,9.7,384186.0
3,2713.8,5.0,5.0,6.0,9.62,4.0,6395.0,9.9,499578.0
4,1659.5,2.0,3.0,17.2,0.50,1.9,3432.0,7.7,269964.0


In [37]:
df["bathrooms"].nunique()

6

In [38]:
df.isna().sum()

,0
area_sqft,28
bedrooms,23
bathrooms,30
age_years,46
distance_km,33
garage_size,69
lot_size_sqft,37
quality_score,49
price,0


In [46]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [40]:
X = df.drop("price", axis=1)
Y = df["price"]

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=42, test_size=0.2)

In [49]:
transformation_simple = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler())
])

In [54]:
preprocessing_simple = ColumnTransformer([
    ("preprocessing", transformation_simple, X.columns)
])

In [55]:
model_simple = Pipeline([
    ("preprocessing", preprocessing_simple),
    ("model", LinearRegression())
])

In [56]:

model_simple.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('preprocessing',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['area_sqft', 'bedrooms', 'bathrooms', 'age_years', 'distance_km',
       'garage_size', 'lot_size_sqft', 'quality_score'],
      dtype='object'))])),
                ('model', LinearRegression())])

In [57]:
preds_simple = model_simple.predict(X_test)

mae_simple = mean_absolute_error(y_test, preds_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, preds_simple))
r2_simple = r2_score(y_test, preds_simple)

print("SimpleImputer + LinearRegression")
print(f"MAE  : {mae_simple:,.2f}")
print(f"RMSE : {rmse_simple:,.2f}")
print(f"R2   : {r2_simple:.4f}")

SimpleImputer + LinearRegression
MAE  : 16,646.53
RMSE : 28,789.02
R2   : 0.9195


# KNN Imputer Pipeline

In [70]:
transformation_knn = Pipeline(
    [("imputer", KNNImputer(n_neighbors=5)), ("scaler", StandardScaler())]
)

In [71]:
preprocessing_knn = ColumnTransformer([
    ("preprocessing", transformation_knn, X.columns)
])

In [72]:
model_knn = Pipeline([
    ("preprocessing", preprocessing_knn),
    ("model", LinearRegression())
])

In [73]:
model_knn.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('preprocessing',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['area_sqft', 'bedrooms', 'bathrooms', 'age_years', 'distance_km',
       'garage_size', 'lot_size_sqft', 'quality_score'],
      dtype='object'))])),
                ('model', LinearRegression())])

In [74]:
preds_knn = model_knn.predict(X_test)

mae_knn = mean_absolute_error(y_test, preds_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test, preds_knn))
r2_knn = r2_score(y_test, preds_knn)

print("KNNImputer (k=5) + LinearRegression")
print(f"MAE  : {mae_knn:,.2f}")
print(f"RMSE : {rmse_knn:,.2f}")
print(f"R2   : {r2_knn:.4f}")

KNNImputer (k=5) + LinearRegression
MAE  : 15,520.78
RMSE : 24,187.68
R2   : 0.9432


In [75]:
results = pd.DataFrame([
    {"Model": "SimpleImputer + LinearRegression", "MAE": mae_simple, "RMSE": rmse_simple, "R2": r2_simple},
    {"Model": "KNNImputer (k=5) + LinearRegression", "MAE": mae_knn, "RMSE": rmse_knn, "R2": r2_knn},
]).sort_values("RMSE")

results

,Model,MAE,RMSE,R2
1,KNNImputer (k=5) + LinearRegression,15520.776451,24187.675929,0.943158
0,SimpleImputer + LinearRegression,16646.525035,28789.015153,0.919475
